# Notebook 3: Monte Carlo Risk and Loss

Part of *Quantifying AI Risk*. Hour 3 of 3. The final layer of the pipeline.

## Setup

NumPy for vectorized sampling. SciPy for the correlation-aware multivariate normal distribution. If SciPy is unavailable, the engine falls back to independent sampling and the main results still hold, but correlated sampling is the right default for governance risk because pillars do not fail independently in practice.

In [1]:
import random
import numpy as np
from scipy.stats import multivariate_normal, norm, lognorm

random.seed(42)
np.random.seed(42)

print("Ready.")

Ready.


## Load the pillar posteriors

Notebook 2 wrote the pillar posteriors to `data/pillar_posteriors.json`. This notebook reads from that file, the same way a real production pipeline would. Hour 1 emits and persists. Hour 2 reads, scores, and persists. Hour 3 reads and simulates. Each stage hands off cleanly to the next.

If the file does not exist because Notebook 2 has not been run yet, the code below falls back to the hardcoded posteriors that Notebook 2 produces after running the full thousand-event stream with drift injection. That keeps this notebook runnable on its own for learners who want to skip ahead. The production-style read-from-disk path is what runs first.

In [2]:
import json
from pathlib import Path

INPUT_PATH = Path("../data/pillar_posteriors.json") if Path("../data/pillar_posteriors.json").exists() else Path("data/pillar_posteriors.json")

if INPUT_PATH.exists():
    with open(INPUT_PATH) as f:
        payload = json.load(f)
    PILLAR_POSTERIORS = payload["pillar_state"]
    print(f"Loaded pillar posteriors from {INPUT_PATH.resolve()}")
    print(f"Source: {payload['n_events']} telemetry events processed by Notebook 2")
else:
    # Fallback: hardcoded posteriors that Notebook 2 produces with the standard
    # 1000-event stream and drift injection. Same numbers, useful for skipping ahead.
    print(f"No posteriors file found at {INPUT_PATH}. Using hardcoded values from a reference run.")
    PILLAR_POSTERIORS = {
        "Fairness & Societal Impact":   {"alpha": 1586.0, "beta": 1277.0},
        "Data & Model Integrity":       {"alpha": 1102.0, "beta":  908.0},
        "Reliability & Robustness":     {"alpha":  541.0, "beta":  469.0},
        "Operational Health":           {"alpha":  375.5, "beta":  134.5},
        "Resilience & Continuity":      {"alpha":  370.5, "beta":  139.5},
    }

print("\nPillar posteriors:\n")
print(f"  {'Pillar':32s}  {'Mean':>8s}  {'Evidence':>10s}")
print(f"  {'-'*32}  {'-'*8}  {'-'*10}")
for pillar, p in PILLAR_POSTERIORS.items():
    mean = p["alpha"] / (p["alpha"] + p["beta"])
    evidence = p["alpha"] + p["beta"] - 10  # subtract prior alpha+beta = 2+8 = 10
    print(f"  {pillar:32s}  {mean:>8.3f}  {evidence:>10.0f}")

Loaded pillar posteriors from /Users/suneetamodekurty2o/Downloads/quantifying-ai-risk-main/data/pillar_posteriors.json
Source: 1000 telemetry events processed by Notebook 2

Pillar posteriors:

  Pillar                                Mean    Evidence
  --------------------------------  --------  ----------
  Operational Health                   0.736         500
  Fairness & Societal Impact           0.554        2853
  Resilience & Continuity              0.726         500
  Data & Model Integrity               0.548        2000
  Reliability & Robustness             0.536        1000


## Map posterior to incident probability

The pillar posterior is a trust score between 0 and 1. The Monte Carlo simulation needs a different input. It needs the probability that an incident occurs in a given month, per pillar.

The mapping from trust score to incident probability is a design choice. The function below uses a logistic-style shape. When the posterior mean is near 1.0, the incident probability per month is near zero. When the posterior mean is near 0.0, the incident probability is at its maximum. In between, the function transitions smoothly.

The specific numbers in this function are illustrative. In production, the mapping is calibrated against the organization's own historical incident rate, recorded per pillar. If historical data is not yet available, start with the defaults below, log every incident as it occurs, and recalibrate the function each quarter.

In [3]:
def posterior_to_incident_prob(posterior_mean, max_prob=0.15, steepness=6.0):
    """
    Maps a posterior trust score to a per-month incident probability.
    Posterior mean of 1.0 -> ~0 probability. Posterior mean of 0.0 -> max_prob.
    """
    # Logistic curve centered at 0.5
    return max_prob / (1 + np.exp(steepness * (posterior_mean - 0.5)))


print("Incident probability calibration:\n")
print(f"  {'Posterior Mean':>16s}  {'Per-month Incident Prob':>24s}")
print(f"  {'-'*16}  {'-'*24}")
for trust in [0.1, 0.3, 0.5, 0.7, 0.9]:
    p = posterior_to_incident_prob(trust)
    print(f"  {trust:>16.2f}  {p:>24.4f}")

print("\nA pillar with trust of 0.3 has ~10% monthly chance of an incident.")
print("A pillar with trust of 0.9 has ~0.4% monthly chance.")

Incident probability calibration:

    Posterior Mean   Per-month Incident Prob
  ----------------  ------------------------
              0.10                    0.1375
              0.30                    0.1153
              0.50                    0.0750
              0.70                    0.0347
              0.90                    0.0125

A pillar with trust of 0.3 has ~10% monthly chance of an incident.
A pillar with trust of 0.9 has ~0.4% monthly chance.


## Loss distributions per pillar

When an incident occurs, the next question is how much it costs. The cost of governance incidents is heavily skewed. Most incidents are small. A small number are catastrophic. The shape of this distribution is what statisticians call *heavy-tailed*.

The engine uses *lognormal distributions* to model the cost per incident, per pillar. A lognormal distribution is the right choice for heavy-tailed positive quantities like dollar losses, because it captures the long right tail without requiring the modeler to enumerate every possible failure scenario.

The parameters below are illustrative and reflect rough industry patterns. Fairness incidents in regulated decisioning carry a high expected cost and a very heavy tail, because of class-action exposure. Operational incidents are smaller and more predictable. In a real deployment, these parameters are derived from the organization's incident history, insurance data, and legal counsel's exposure assessments.

In [4]:
# Lognormal parameters per pillar: (mu, sigma) of the underlying normal.
# Median loss = exp(mu). Tail heaviness grows with sigma.
LOSS_DISTRIBUTIONS = {
    "Fairness & Societal Impact":   {"mu": np.log(500_000),  "sigma": 1.8},
    "Data & Model Integrity":       {"mu": np.log(200_000),  "sigma": 1.5},
    "Reliability & Robustness":     {"mu": np.log(100_000),  "sigma": 1.3},
    "Operational Health":           {"mu": np.log( 50_000),  "sigma": 1.0},
    "Resilience & Continuity":      {"mu": np.log( 75_000),  "sigma": 1.0},
}

print("Loss distribution shapes (if incident occurs):\n")
print(f"  {'Pillar':32s}  {'Median $':>12s}  {'Mean $':>14s}  {'P95 $':>14s}  {'P99 $':>14s}")
print(f"  {'-'*32}  {'-'*12}  {'-'*14}  {'-'*14}  {'-'*14}")

for pillar, params in LOSS_DISTRIBUTIONS.items():
    mu, sigma = params["mu"], params["sigma"]
    median = np.exp(mu)
    mean = np.exp(mu + sigma**2 / 2)
    p95 = np.exp(mu + sigma * norm.ppf(0.95))
    p99 = np.exp(mu + sigma * norm.ppf(0.99))
    print(f"  {pillar:32s}  {median:>12,.0f}  {mean:>14,.0f}  {p95:>14,.0f}  {p99:>14,.0f}")

print("\nNotice the P99 is often 5-10x the median. That is the tail that")
print("classical expected-loss calculations miss.")

Loss distribution shapes (if incident occurs):

  Pillar                                Median $          Mean $           P95 $           P99 $
  --------------------------------  ------------  --------------  --------------  --------------
  Fairness & Societal Impact             500,000       2,526,545       9,656,095      32,926,539
  Data & Model Integrity                 200,000         616,043       2,358,068       6,554,055
  Reliability & Robustness               100,000         232,798         848,508       2,057,861
  Operational Health                      50,000          82,436         259,013         512,024
  Resilience & Continuity                 75,000         123,654         388,519         768,036

Notice the P99 is often 5-10x the median. That is the tail that
classical expected-loss calculations miss.


## Correlation between pillars

This is the part most simple risk models skip, and it is where the real tail risk lives. In production, when one governance pillar fails, others usually fail with it, because they are all responding to the same underlying event. We saw this in Hour 1, when drift arrived and every signal degraded together.

A *correlation matrix* expresses the strength of these joint movements. A value of 1.0 between two pillars means they always move together. A value of 0.0 means they move independently. The matrix below reflects the pattern from the streaming scenario in Hour 1. Fairness and data integrity are strongly correlated. Operational health and resilience are strongly correlated. Cross-group correlations are moderate.

The simulation samples correlated standard normal values, then transforms them into both the incident indicator and the loss scale. This way, when one pillar's incident triggers, the others are more likely to trigger as well, and the losses themselves are more likely to be large at the same time. That joint behavior is what makes the tail of the loss distribution heavy. A simulation that assumes independence between pillars underestimates the tail, often by a factor of three to five.

In [5]:
PILLAR_ORDER = [
    "Fairness & Societal Impact",
    "Data & Model Integrity",
    "Reliability & Robustness",
    "Operational Health",
    "Resilience & Continuity",
]

# Correlation matrix calibrated from the Hour 1 streaming scenario.
# Illustrative. In production, compute this from your actual signal correlation.
CORRELATION_MATRIX = np.array([
    # Fair  Data  Rel   Ops   Res
    [1.00, 0.70, 0.50, 0.30, 0.30],  # Fairness
    [0.70, 1.00, 0.55, 0.35, 0.35],  # Data Integrity
    [0.50, 0.55, 1.00, 0.45, 0.45],  # Reliability
    [0.30, 0.35, 0.45, 1.00, 0.75],  # Operational Health
    [0.30, 0.35, 0.45, 0.75, 1.00],  # Resilience
])

print("Pillar correlation matrix:\n")
print(f"  {'':32s}  " + "  ".join(f"{p[:4]:>6s}" for p in PILLAR_ORDER))
for i, p in enumerate(PILLAR_ORDER):
    row = f"  {p:32s}  " + "  ".join(f"{CORRELATION_MATRIX[i][j]:>6.2f}" for j in range(5))
    print(row)

print("\nFairness and Data Integrity move together (0.70).")
print("Operational Health and Resilience move together (0.75).")
print("Independent sampling would underestimate joint failure risk.")

Pillar correlation matrix:

                                      Fair    Data    Reli    Oper    Resi
  Fairness & Societal Impact          1.00    0.70    0.50    0.30    0.30
  Data & Model Integrity              0.70    1.00    0.55    0.35    0.35
  Reliability & Robustness            0.50    0.55    1.00    0.45    0.45
  Operational Health                  0.30    0.35    0.45    1.00    0.75
  Resilience & Continuity             0.30    0.35    0.45    0.75    1.00

Fairness and Data Integrity move together (0.70).
Operational Health and Resilience move together (0.75).
Independent sampling would underestimate joint failure risk.


## Run the Monte Carlo simulation

A *Monte Carlo simulation* is a method for estimating the behavior of a complex system by sampling from its inputs many times and recording the outcomes. In this engine, each sample represents one simulated month of the production system.

In each simulated month, for each pillar, the engine samples whether an incident occurs (correlated across pillars), and if so, how much it costs (also correlated across pillars). The losses are summed across pillars to produce the total loss for that month. The simulation runs ten thousand times, generating a distribution of ten thousand simulated monthly outcomes.

The resulting distribution is the answer. Every number that comes after this point in the notebook is derived from that distribution.

In [6]:
N_SIMULATIONS = 10_000


def run_monte_carlo(pillar_posteriors, loss_distributions, correlation_matrix,
                    pillar_order, n_sims=N_SIMULATIONS):

    # Pre-compute per-pillar incident probabilities from posteriors
    incident_probs = np.array([
        posterior_to_incident_prob(
            pillar_posteriors[p]["alpha"] / (pillar_posteriors[p]["alpha"] + pillar_posteriors[p]["beta"])
        )
        for p in pillar_order
    ])

    # Thresholds on the standard normal that correspond to those probabilities
    # If correlated-normal draw > threshold, an incident occurred for that pillar
    incident_thresholds = norm.ppf(1 - incident_probs)

    # Draw correlated standard normals for incident indicators
    incident_draws = multivariate_normal.rvs(
        mean=np.zeros(len(pillar_order)),
        cov=correlation_matrix,
        size=n_sims,
    )
    incident_occurred = incident_draws > incident_thresholds  # shape (n_sims, n_pillars)

    # For each pillar, draw lognormal losses (also correlated, using same matrix)
    loss_draws = multivariate_normal.rvs(
        mean=np.zeros(len(pillar_order)),
        cov=correlation_matrix,
        size=n_sims,
    )

    losses = np.zeros_like(incident_occurred, dtype=float)
    for j, pillar in enumerate(pillar_order):
        mu = loss_distributions[pillar]["mu"]
        sigma = loss_distributions[pillar]["sigma"]
        # Transform standard normal into lognormal loss
        losses[:, j] = np.exp(mu + sigma * loss_draws[:, j])

    # Only incurred when an incident occurred
    incurred_losses = losses * incident_occurred

    # Total per-month loss across pillars
    total_losses = incurred_losses.sum(axis=1)

    return total_losses, incurred_losses


total_losses, per_pillar_losses = run_monte_carlo(
    PILLAR_POSTERIORS, LOSS_DISTRIBUTIONS, CORRELATION_MATRIX, PILLAR_ORDER
)

print(f"Ran {N_SIMULATIONS:,} simulated months.\n")
print(f"Months with at least one incident: {(total_losses > 0).sum():,}")
print(f"Months with zero incidents:        {(total_losses == 0).sum():,}")

Ran 10,000 simulated months.

Months with at least one incident: 1,732
Months with zero incidents:        8,268


## The executive summary

The numbers below are the ones that go on a board slide. Each one answers a specific question.

*Expected monthly loss* is the average across all simulated months. It is the long-run average cost of operating the system.

*Value at Risk* (VaR) at a given percentile is the loss threshold below which most outcomes fall. The 95% VaR is the threshold below which ninety-five percent of months fall. The 99% VaR is the threshold below which ninety-nine percent of months fall. The 99% VaR is always higher than the 95% VaR, because it is a more extreme cutoff.

*Tail Conditional Expectation* (TCE) is the average loss within the worst months. The TCE at 95% is the average loss across the worst five percent of months. Unlike VaR, which only tells you where the bad zone starts, the TCE tells you how bad it actually is once you are in the bad zone.

The TCE is the number that keeps a CFO honest, because it represents what actually happens when things go wrong, rather than the average case. Regulators increasingly expect TCE alongside VaR, because VaR alone underestimates catastrophic exposure.

In [7]:
expected_loss = total_losses.mean()
median_loss = np.median(total_losses)
p95 = np.percentile(total_losses, 95)
p99 = np.percentile(total_losses, 99)
tail_ce_95 = total_losses[total_losses >= p95].mean()
tail_ce_99 = total_losses[total_losses >= p99].mean()

print("Monthly governance risk summary\n")
print(f"  Expected loss (mean):                  ${expected_loss:>14,.0f}")
print(f"  Median loss:                           ${median_loss:>14,.0f}")
print(f"  95% VaR (95th percentile):             ${p95:>14,.0f}")
print(f"  99% VaR (99th percentile):             ${p99:>14,.0f}")
print(f"  Tail CE at 95% (avg of worst 5%):      ${tail_ce_95:>14,.0f}")
print(f"  Tail CE at 99% (avg of worst 1%):      ${tail_ce_99:>14,.0f}")

print("\nThe tail CE is higher than the VaR by design.")
print("VaR tells you where the bad zone starts. Tail CE tells you how bad it")
print("actually is once you are in the bad zone. Regulators increasingly want")
print("both because VaR alone underestimates catastrophic exposure.")

Monthly governance risk summary

  Expected loss (mean):                  $       244,925
  Median loss:                           $             0
  95% VaR (95th percentile):             $       586,364
  99% VaR (99th percentile):             $     4,605,387
  Tail CE at 95% (avg of worst 5%):      $     4,499,418
  Tail CE at 99% (avg of worst 1%):      $    16,181,422

The tail CE is higher than the VaR by design.
VaR tells you where the bad zone starts. Tail CE tells you how bad it
actually is once you are in the bad zone. Regulators increasingly want
both because VaR alone underestimates catastrophic exposure.


## Which pillars drive the tail

Expected loss tells you where the system is on average. *Tail attribution* tells you where catastrophic risk comes from, which is often a different answer.

A pillar can have a low expected loss and still drive a large share of tail risk. This happens when the pillar's loss distribution is heavy-tailed and it correlates with other pillars going bad at the same time. The average behavior is mild. The joint behavior under stress is severe.

Tail attribution is the view that turns Monte Carlo from an interesting number into a remediation plan. The pillars that drive the tail are where governance investment produces the largest reduction in catastrophic exposure. They are not necessarily the pillars with the highest expected loss.

In [8]:
# Identify tail months (worst 5%)
tail_threshold = np.percentile(total_losses, 95)
tail_mask = total_losses >= tail_threshold

# Average per-pillar loss in tail months vs overall
print("Contribution to the worst 5% of months\n")
print(f"  {'Pillar':32s}  {'Avg in tail':>14s}  {'Avg overall':>14s}  {'Tail share':>12s}")
print(f"  {'-'*32}  {'-'*14}  {'-'*14}  {'-'*12}")

tail_totals = per_pillar_losses[tail_mask].mean(axis=0)
overall_totals = per_pillar_losses.mean(axis=0)
total_tail_loss = tail_totals.sum()

for j, pillar in enumerate(PILLAR_ORDER):
    tail_avg = tail_totals[j]
    overall_avg = overall_totals[j]
    share = tail_avg / total_tail_loss * 100
    print(f"  {pillar:32s}  ${tail_avg:>12,.0f}  ${overall_avg:>12,.0f}  {share:>11.1f}%")

print("\nThe pillars with the largest tail share are where governance investment")
print("reduces catastrophic exposure most. Not where average losses are highest.")

Contribution to the worst 5% of months

  Pillar                               Avg in tail     Avg overall    Tail share
  --------------------------------  --------------  --------------  ------------
  Fairness & Societal Impact        $   3,613,892  $     185,899         80.3%
  Data & Model Integrity            $     653,128  $      37,828         14.5%
  Reliability & Robustness          $     172,889  $      14,640          3.8%
  Operational Health                $      20,636  $       2,566          0.5%
  Resilience & Continuity           $      38,873  $       3,993          0.9%

The pillars with the largest tail share are where governance investment
reduces catastrophic exposure most. Not where average losses are highest.


## Sensitivity: which assumption matters most

Every number above depends on assumptions. Incident probability calibration, loss distribution parameters, correlation strength. A regulator, a CFO, or an auditor will ask which of those assumptions the result is most sensitive to.

*Sensitivity analysis* answers that question by re-running the simulation with one assumption tweaked at a time and recording how much the result changes. An assumption that moves the result by twenty percent when tweaked is one the organization should invest in measuring more precisely. An assumption that barely moves the result can stay at its default.

This is the analysis that defends the methodology under audit. The auditor will not accept "we assumed this is true." The auditor will accept "we tested how much the answer depends on this assumption, and the result was insensitive within reasonable ranges." Sensitivity analysis turns assumptions into bounded uncertainty.

In [9]:
def sensitivity_run(tweak_fn, label):
    """Run the simulation with one tweak applied and report the delta."""
    # Copy and tweak
    posteriors = {p: dict(v) for p, v in PILLAR_POSTERIORS.items()}
    losses = {p: dict(v) for p, v in LOSS_DISTRIBUTIONS.items()}
    corr = CORRELATION_MATRIX.copy()
    tweak_fn(posteriors, losses, corr)

    totals, _ = run_monte_carlo(posteriors, losses, corr, PILLAR_ORDER, n_sims=5_000)
    new_expected = totals.mean()
    new_p99 = np.percentile(totals, 99)
    return label, new_expected, new_p99


print("Sensitivity analysis (expected loss, 99% VaR)\n")
baseline_exp = expected_loss
baseline_p99 = p99
print(f"  {'Scenario':48s}  {'Exp Loss':>12s}  {'99% VaR':>12s}")
print(f"  {'-'*48}  {'-'*12}  {'-'*12}")
print(f"  {'baseline':48s}  ${baseline_exp:>10,.0f}  ${baseline_p99:>10,.0f}")

scenarios = [
    ("fairness loss median +50%",
     lambda p, l, c: l["Fairness & Societal Impact"].update(mu=np.log(750_000))),
    ("all correlations set to 0 (naive independence)",
     lambda p, l, c: c.__setitem__(slice(None), np.eye(5))),
    ("fairness posterior improved (alpha +2000)",
     lambda p, l, c: p["Fairness & Societal Impact"].update(alpha=p["Fairness & Societal Impact"]["alpha"] + 2000)),
    ("data integrity loss tail heavier (sigma +0.5)",
     lambda p, l, c: l["Data & Model Integrity"].update(sigma=2.0)),
]

for label, tweak_fn in scenarios:
    try:
        _, exp, vap99 = sensitivity_run(tweak_fn, label)
        delta_exp_pct = (exp - baseline_exp) / baseline_exp * 100
        delta_p99_pct = (vap99 - baseline_p99) / baseline_p99 * 100
        print(f"  {label:48s}  ${exp:>10,.0f}  ${vap99:>10,.0f}   ({delta_exp_pct:+.0f}% / {delta_p99_pct:+.0f}%)")
    except Exception as e:
        print(f"  {label:48s}  error: {e}")

print("\nThe assumption the result is most sensitive to is where you invest")
print("in better measurement. An assumption the result is insensitive to can")
print("stay at default. This is the core insight of sensitivity analysis.")

Sensitivity analysis (expected loss, 99% VaR)

  Scenario                                              Exp Loss       99% VaR
  ------------------------------------------------  ------------  ------------
  baseline                                          $   244,925  $ 4,605,387
  fairness loss median +50%                         $   381,748  $ 6,097,018   (+56% / +32%)
  all correlations set to 0 (naive independence)    $   178,521  $ 3,297,588   (-27% / -28%)
  fairness posterior improved (alpha +2000)         $   124,012  $ 1,763,731   (-49% / -62%)
  data integrity loss tail heavier (sigma +0.5)     $   249,696  $ 5,172,417   (+2% / +12%)

The assumption the result is most sensitive to is where you invest
in better measurement. An assumption the result is insensitive to can
stay at default. This is the core insight of sensitivity analysis.


## The shape of the distribution

Summary numbers tell one story. The shape of the underlying distribution tells another. An executive who sees a histogram of monthly loss understands something that the same executive staring at a single expected loss number cannot get. The shape shows how often the tail actually arrives, and how heavy it is when it does.

The text-based histogram below works in any environment. If matplotlib is available, the same data can be plotted as a standard bar chart for sharper visual impact.

In [10]:
# Text histogram for environments without matplotlib
bins = [0, 50_000, 100_000, 250_000, 500_000, 1_000_000, 2_000_000, 5_000_000, float('inf')]
bin_labels = ["$0", "$0-50K", "$50-100K", "$100-250K", "$250-500K", "$500K-1M", "$1-2M", "$2-5M", "$5M+"]

# For 'zero loss' bucket, count separately
zero_count = (total_losses == 0).sum()
print("Distribution of monthly loss\n")
print(f"  {'Range':14s}  {'Count':>8s}  {'Share':>8s}  Histogram")
print(f"  {'-'*14}  {'-'*8}  {'-'*8}  {'-'*40}")
print(f"  {'No incident':14s}  {zero_count:>8,}  {zero_count/N_SIMULATIONS*100:>7.1f}%  {'#' * int(zero_count/N_SIMULATIONS*40)}")

nonzero = total_losses[total_losses > 0]
for i in range(1, len(bins)):
    lo, hi = bins[i-1], bins[i]
    count = ((nonzero > lo) & (nonzero <= hi)).sum()
    share = count / N_SIMULATIONS * 100
    bar = '#' * max(1, int(share * 2)) if count > 0 else ''
    print(f"  {bin_labels[i]:14s}  {count:>8,}  {share:>7.1f}%  {bar}")

print(f"\n  Worst month in simulation: ${total_losses.max():,.0f}")

Distribution of monthly loss

  Range              Count     Share  Histogram
  --------------  --------  --------  ----------------------------------------
  No incident        8,268     82.7%  #################################
  $0-50K               330      3.3%  ######
  $50-100K             237      2.4%  ####
  $100-250K            365      3.6%  #######
  $250-500K            261      2.6%  #####
  $500K-1M             190      1.9%  ###
  $1-2M                151      1.5%  ###
  $2-5M                107      1.1%  ##
  $5M+                  91      0.9%  #

  Worst month in simulation: $233,575,345


## Your turn: reserve calculation and business decision

A Monte Carlo risk engine is only useful if its outputs drive a decision. The exercise below asks the learner to translate the numbers from this notebook into three specific decisions that a CFO and an AI governance committee can act on.

What to do:

1. **Reserve calculation.** Decide what reserve amount the company should hold for AI governance incidents in the coming quarter. Justify whether the reserve is set at the expected loss, the 95% VaR, the 99% VaR, or some other figure. There is no single right answer, but there is a defensible answer and an indefensible one.

2. **Remediation priority.** Based on the tail attribution, rank the pillars in order of where remediation investment produces the greatest reduction in catastrophic exposure. This ranking is different from a ranking by expected loss, and recognizing that difference is the point of the exercise.

3. **Telemetry investment.** Based on the sensitivity analysis, identify the one area where improving measurement, through better telemetry or better calibration, would most reduce uncertainty in the risk number itself. That is where the governance engineering team should invest next.

This notebook does not provide a solution. In production, these decisions involve judgment calls about risk tolerance, regulatory scope, and business strategy. The Monte Carlo output informs the decision but does not make it. Practice the judgment call now, in writing, before the stakes are real.

In [11]:
# Your analysis space
# Use the variables already in scope:
#   expected_loss, median_loss, p95, p99, tail_ce_95, tail_ce_99
#   per_pillar_losses, tail_mask
#   PILLAR_ORDER, PILLAR_POSTERIORS, LOSS_DISTRIBUTIONS

# Print the numbers you need, then write your analysis in the markdown cell below


### Your recommendations

*This is the cell referenced in the exercise above. Write your recommendations here.*

*The kind of memo a governance committee or a CFO reads is short, structured, and defensible. The math has been done in the notebook. The writing is what carries the math into a decision.*

*Three to five sentences per item. Address all three:*

*1. The reserve amount you recommend and your justification.*
*2. The remediation priority ranking and the reasoning behind it.*
*3. The top telemetry investment and the expected impact of making it.*

*Writing this kind of memo clearly is as important as the math that produced the numbers. Treat the exercise accordingly.*

## Generate your report

Everything built across three hours collapses into a single one-page document. Run the cell below. The pipeline writes a PDF to your local folder.

This is the artifact. The expected loss number, the pillar scores, the loss distribution, the top risk drivers, the regulatory mapping. All on one page. This is the document a CFO would accept, a board would attach to a memo, or a regulator would treat as evidence.

The methodology produces a deliverable, not just a number.

In [12]:
import sys
from pathlib import Path

# Add the repo root to the Python path so `utils` is importable
# from inside the notebooks/ folder
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from utils.report_generator import generate_report

results = {
    "expected_loss": expected_loss,
    "median_loss": median_loss,
    "p95": p95,
    "p99": p99,
    "tail_ce_95": tail_ce_95,
    "tail_ce_99": tail_ce_99,
    "total_losses": total_losses,
    "per_pillar_losses": per_pillar_losses,
    "tail_mask": tail_mask,
    "pillar_order": PILLAR_ORDER,
    "pillar_posteriors": PILLAR_POSTERIORS,
    "n_simulations": N_SIMULATIONS,
    "model_id": "credit_risk_v3.2",
    "reporting_period": "April 2026",
}

report_path = generate_report(results)
print(f"Report saved to: {report_path}")
print("\nOpen the file in your folder to see your one-page risk assessment.")

Report saved to: ai_risk_report_20260524_024434.pdf

Open the file in your folder to see your one-page risk assessment.


## What this notebook produced

A Monte Carlo risk engine that takes pillar posteriors, models incident probabilities and loss distributions with an explicit correlation structure, runs ten thousand simulated months, and produces expected loss, value at risk, tail conditional expectation, tail attribution, sensitivity analysis, and a readable distribution view.

Below is the full stack you have built across three hours:

- **Hour 1 (telemetry).** Raw signals emitted from the model, structured, continuous, mapped to governance dimensions.
- **Hour 2 (Bayesian scoring).** Signals rolled up into pillar posteriors with credible intervals, using a skeptical prior and severity-weighted evidence.
- **Hour 3 (Monte Carlo risk).** Posteriors propagated into a probability distribution of financial outcomes, with tail attribution for remediation prioritization.

Each layer transforms the layer below it into what the next audience needs. Engineers work at Hour 1. Compliance works at Hour 2. Executives work at Hour 3. All three views share the same underlying evidence, which means the same incident affects all three views consistently. When engineers see drift in the telemetry, compliance sees the trust score drop, and executives see the reserve requirement rise. Nobody is working from a different version of reality.

The implementation choices in these notebooks are deliberately simple so the methodology stays visible. A production deployment will have different pillars, different severity weights, different loss distributions, and a different correlation structure. The math stays the same. The architecture is what generalizes.